In [ ]:
# Видаляємо стару копію (якщо вже клонували раніше)
!rm -rf deep

# Клонуємо особистий репозиторій з конспектами і матеріалами
# (репозиторій курсу goitacademy — лише для довідки, всі матеріали вже у PetraStill/deep)
!git clone https://github.com/PetraStill/deep.git

# Переходимо в директорію репозиторію
%cd deep

In [ ]:
# Встановлюємо бібліотеку ultralytics для роботи з YOLO
# --upgrade гарантує встановлення останньої версії
!pip install --upgrade ultralytics

# ray[tune] — бібліотека для розподіленого підбору гіперпараметрів (потрібна в темі 14)
!pip install --upgrade -U "ray[tune]"

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # Вимикаємо попередження — щоб вивід був чистішим

import os       # Робота з файловою системою (шляхи, папки)
import re       # Регулярні вирази
import glob     # Пошук файлів за маскою
import random   # Генерація випадкових чисел (для відтворюваності)
import yaml     # Читання і запис YAML-конфігурацій

import numpy as np           # Числові операції, масиви
import pandas as pd          # Табличні дані, DataFrame

import matplotlib.pyplot as plt          # Візуалізація
from matplotlib.patches import Rectangle # Малювання прямокутників на графіках
import seaborn as sns                    # Розширена статистична візуалізація

from PIL import Image  # Завантаження і обробка зображень (Python Imaging Library)
import cv2             # OpenCV — зчитування, запис та обробка зображень

from ultralytics import YOLO  # Клас для завантаження і тренування YOLO-моделей

# Відображати графіки inline (у Jupyter/Colab)
%matplotlib inline

# Вимикаємо логування Weights & Biases (якщо wandb встановлено)
!wandb disabled

In [ ]:
class CFG:
    """
    Клас-контейнер для всіх гіперпараметрів і налаштувань тренування.
    Зберігання конфігурацій в одному місці полегшує відтворюваність і зміну параметрів.
    """

    # --- Режим дебагу ---
    DEBUG = False  # True: швидкі тести на малій частині датасету; False: повне тренування
    FRACTION = 0.05 if DEBUG else 1.0  # Частка датасету для тренування (5% у debug)
    SEED = 42  # Зафіксований seed для відтворюваності результатів

    # --- Класи ---
    CLASSES = [
        'Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask',
        'NO-Safety Vest', 'Person', 'Safety Cone',
        'Safety Vest', 'machinery', 'vehicle'
    ]  # Список назв класів у порядку відповідно до анотацій
    NUM_CLASSES_TO_TRAIN = len(CLASSES)  # 10 — кількість класів

    # --- Параметри тренування ---
    EPOCHS = 3 if DEBUG else 70  # Кількість епох; 70 — для повного тренування
    BATCH_SIZE = 8   # Кількість зображень в одному батчі; 16 при більшій GPU-пам'яті

    # --- Модель ---
    BASE_MODEL = 'yolov9e'  # Вибір архітектури: yolov8n/s/m/l/x, yolov9c/e
    BASE_MODEL_WEIGHTS = f'{BASE_MODEL}.pt'  # Файл ваг: автозавантажується при першому запуску
    EXP_NAME = f'ppe_css_{EPOCHS}_epochs'  # Назва експерименту для збереження результатів

    # --- Оптимізатор та навчання ---
    OPTIMIZER = 'auto'     # auto: ultralytics обирає оптимізатор; або SGD/Adam/AdamW тощо
    LR = 1e-3              # Початковий learning rate (lr0)
    LR_FACTOR = 0.01       # Кінцевий lr = LR * LR_FACTOR → поступове зменшення
    WEIGHT_DECAY = 5e-4    # L2-регуляризація: штраф за великі ваги (запобігає overfitting)
    DROPOUT = 0.0          # Dropout rate: 0.0 — вимкнено
    PATIENCE = 20          # Early stopping: зупинити якщо 20 епох без покращення
    PROFILE = False        # Профілювання швидкості шарів (для дебагу)
    LABEL_SMOOTHING = 0.0  # Label smoothing: розм'якшення міток (0.0 — вимкнено)

     # --- Шляхи ---
    # Автоматично визначаємо середовище виконання:
    #   Colab:  після `git clone PetraStill/deep && %cd deep` → датасет у ./тема_7/
    #   Kaggle: після `!git clone PetraStill/deep` в /kaggle/working/ → датасет у ./deep/тема_7/
    #   Локально: шлях до папки тема_7 у клонованому репо
    CUSTOM_DATASET_DIR = './тема_7/'      # Colab / локально
    # CUSTOM_DATASET_DIR = './deep/тема_7/'  # Kaggle (розкоментуй якщо запускаєш на Kaggle)
    OUTPUT_DIR = './'  # Куди зберігати data.yaml і результати тренування

In [ ]:
# Формуємо словник з усіма необхідними полями для YOLO
dict_file = {
    'train': os.path.join(CFG.CUSTOM_DATASET_DIR, 'train'),  # Шлях до train зображень
    'val':   os.path.join(CFG.CUSTOM_DATASET_DIR, 'valid'),  # Шлях до validation зображень
    'test':  os.path.join(CFG.CUSTOM_DATASET_DIR, 'test'),   # Шлях до test зображень
    'nc':    CFG.NUM_CLASSES_TO_TRAIN,                        # Кількість класів (nc = num_classes)
    'names': CFG.CLASSES                                      # Список назв класів
}

# Записуємо словник у файл data.yaml у директорії виводу
with open(os.path.join(CFG.OUTPUT_DIR, 'data.yaml'), 'w+') as file:
    yaml.dump(dict_file, file)

In [ ]:
# Функція для зчитування YAML файлу
def read_yaml_file(file_path=CFG.CUSTOM_DATASET_DIR):
    with open(file_path, 'r') as file:
        try:
            data = yaml.safe_load(file)  # safe_load — безпечна альтернатива yaml.load()
            return data
        except yaml.YAMLError as e:
            print("Error reading YAML:", e)
            return None

# Функція для красивого виведення YAML
def print_yaml_data(data):
    formatted_yaml = yaml.dump(data, default_style=False)  # default_style=False → читабельний формат
    print(formatted_yaml)

# Зчитуємо і виводимо створений файл
file_path = os.path.join(CFG.OUTPUT_DIR, 'data.yaml')
yaml_data = read_yaml_file(file_path)

if yaml_data:
    print_yaml_data(yaml_data)

In [ ]:
def display_image(image, print_info=True, hide_axis=False):
    """
    Відображає зображення — приймає як шлях до файлу, так і NumPy-масив.

    image:      str (шлях) або np.ndarray (масив пікселів)
    print_info: bool — виводити тип і розмір зображення
    hide_axis:  bool — приховати осі координат на графіку
    """
    if isinstance(image, str):       # Якщо передано шлях до файлу
        img = Image.open(image)       # Відкриваємо зображення через Pillow
        plt.imshow(img)               # Відображаємо на графіку
    elif isinstance(image, np.ndarray):  # Якщо передано масив NumPy (наприклад, від OpenCV)
        image = image[..., ::-1]      # Конвертуємо BGR → RGB (OpenCV зберігає в BGR!)
        img = Image.fromarray(image)  # Перетворюємо масив у PIL Image
        plt.imshow(img)
    else:
        raise ValueError("Unsupported image format")  # Невідомий формат

    if print_info:
        print('Type: ', type(img), '\n')
        print('Shape: ', np.array(img).shape, '\n')  # (висота, ширина, канали)

    if hide_axis:
        plt.axis('off')  # Вимикаємо осі — для чистішого відображення

    plt.show()


# Відображаємо конкретне тренувальне зображення
example_image_path = (
    CFG.CUSTOM_DATASET_DIR +
    'train/images/-2297-_png_jpg.rf.9fff3740d864fbec9cda50d783ad805e.jpg'
)
display_image(example_image_path, print_info=True, hide_axis=False)

In [ ]:
def get_image_properties(image_path):
    """
    Повертає словник із властивостями зображення.

    image_path: str — шлях до файлу зображення
    Повертає: dict з ключами width, height, channels, dtype
    """
    img = cv2.imread(image_path)  # cv2 зчитує зображення як NumPy масив BGR

    if img is None:
        raise ValueError("Could not read image file")  # Файл не знайдено або пошкоджено

    properties = {
        "width":    img.shape[1],                                 # Ширина в пікселях
        "height":   img.shape[0],                                 # Висота в пікселях
        "channels": img.shape[2] if len(img.shape) == 3 else 1,  # 3 для RGB, 1 для grayscale
        "dtype":    img.dtype,                                    # Тип даних (uint8: 0–255)
    }
    return properties

img_properties = get_image_properties(example_image_path)
print(img_properties)

In [ ]:
def plot_random_images_from_folder(folder_path, num_images=20, seed=CFG.SEED):
    """
    Відображає сітку випадково обраних зображень з теки.

    folder_path: str — шлях до теки із зображеннями
    num_images:  int — скільки зображень показати
    seed:        int — seed для відтворюваності вибірки
    """
    random.seed(seed)  # Фіксуємо seed — завжди отримуємо однакову вибірку

    # Збираємо всі файли зображень у теці за відомими розширеннями
    image_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(('.jpg', '.png', '.jpeg', '.gif'))
    ]

    if len(image_files) < num_images:
        raise ValueError("Not enough images in the folder")

    # Випадково обираємо num_images файлів (без повторення)
    selected_files = random.sample(image_files, num_images)

    # Розраховуємо розміри сітки: 5 стовпців, решта — рядки
    num_cols = 5
    num_rows = (num_images + num_cols - 1) // num_cols  # Округлення вгору
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 8))

    for i, file_name in enumerate(selected_files):
        img = Image.open(os.path.join(folder_path, file_name))

        # Вибираємо правильну вісь залежно від кількості рядків
        if num_rows == 1:
            ax = axes[i % num_cols]
        else:
            ax = axes[i // num_cols, i % num_cols]

        ax.imshow(img)
        ax.axis('off')  # Прибираємо осі — зображення без зайвих елементів

    # Видаляємо порожні підграфіки (якщо кількість не кратна num_cols)
    for i in range(num_images, num_rows * num_cols):
        if num_rows == 1:
            fig.delaxes(axes[i % num_cols])
        else:
            fig.delaxes(axes[i // num_cols, i % num_cols])

    plt.tight_layout()  # Автоматично підлаштовує відступи між підграфіками
    plt.show()


# Показуємо 20 випадкових зображень з тренувальної вибірки
plot_random_images_from_folder(
    folder_path=CFG.CUSTOM_DATASET_DIR + 'train/images/',
    num_images=20
)

In [ ]:
# Словник: рядковий індекс → назва класу
class_idx = {str(i): CFG.CLASSES[i] for i in range(CFG.NUM_CLASSES_TO_TRAIN)}

class_stat = {}   # Статистика по класах для кожної вибірки
data_len = {}     # Кількість файлів у кожній вибірці
class_info = []   # Зберігаємо для DataFrame

for mode in ['train', 'valid', 'test']:
    # Ініціалізуємо лічильник по кожному класу
    class_count = {CFG.CLASSES[i]: 0 for i in range(CFG.NUM_CLASSES_TO_TRAIN)}

    path = os.path.join(CFG.CUSTOM_DATASET_DIR, mode, 'labels')

    for file in os.listdir(path):
        with open(os.path.join(path, file)) as f:
            lines = f.readlines()  # Кожна рядок = один об'єкт на зображенні

            # line[0] — перший символ рядка = клас (YOLO формат: клас x y w h)
            # set() — уникаємо повторного рахунку одного класу двічі на одному зображенні
            for cls in set([line[0] for line in lines]):
                class_count[class_idx[cls]] += 1

    data_len[mode] = len(os.listdir(path))
    class_stat[mode] = class_count
    class_info.append({'Mode': mode, **class_count, 'Data_Volume': data_len[mode]})

# Зберігаємо у DataFrame для зручного аналізу
dataset_stats_df = pd.DataFrame(class_info)

with pd.option_context('display.max_columns', None):  # Показуємо всі стовпці
    display(dataset_stats_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # 3 графіки в ряд (train/valid/test)

for i, mode in enumerate(['train', 'valid', 'test']):
    sns.barplot(
        data=dataset_stats_df[dataset_stats_df['Mode'] == mode].drop(columns='Mode'),
        orient='v',      # Вертикальні стовпці
        ax=axes[i],      # Малюємо на i-й осі
        palette='Set2'   # Палітра кольорів
    )

    axes[i].set_title(f'{mode.capitalize()} Class Statistics')  # Заголовок
    axes[i].set_xlabel('Classes')   # Підпис осі X
    axes[i].set_ylabel('Count')     # Підпис осі Y
    axes[i].tick_params(axis='x', rotation=90)  # Назви класів вертикально (не перекриваються)

    # Додаємо числові підписи над кожним стовпцем
    for p in axes[i].patches:
        axes[i].annotate(
            f"{int(p.get_height())}",  # Значення: висота стовпця = кількість
            (p.get_x() + p.get_width() / 2., p.get_height()),  # Позиція підпису
            ha='center', va='center', fontsize=8,
            color='black', xytext=(0, 5), textcoords='offset points'
        )

plt.tight_layout()
plt.show()

In [ ]:
# Завантажуємо попередньо навчену модель YOLOv9e
# При першому запуску файл .pt автоматично завантажується з ultralytics servers
model = YOLO(CFG.BASE_MODEL_WEIGHTS)

# Запускаємо inference на тестовому зображенні
results = model.predict(
    source=example_image_path,  # Шлях до вхідного зображення
    classes=[0],                # Детектуємо тільки клас 0: Person (з COCO)
    conf=0.30,                  # Поріг впевненості: відкидаємо рамки з confidence < 0.30
    device=None,                # None = CPU; [0] = перша GPU; [0, 1] = дві GPU
    imgsz=(                     # Розмір входу для моделі (висота, ширина)
        img_properties['height'],
        img_properties['width']
    ),
    save=True,                  # Зберегти зображення з намальованими рамками
    save_txt=True,              # Зберегти анотації у форматі YOLO .txt
    save_conf=True,             # Додати confidence score до збережених анотацій
    exist_ok=True,              # Не кидати помилку, якщо директорія вже існує
)

# Відображаємо результат inference
example_image_inference_output = example_image_path.split('/')[-1]
display_image(f'runs/detect/predict/{example_image_inference_output}')

In [ ]:
# Завантажуємо базову модель з претренованими вагами COCO
model = YOLO(CFG.BASE_MODEL_WEIGHTS)

In [ ]:
%%time

model.train(
    # --- Дані ---
    data=os.path.join(CFG.OUTPUT_DIR, 'data.yaml'),  # Шлях до конфігурації датасету
    task='detect',  # Задача: 'detect' (bounding boxes), або 'segment', 'classify', 'pose'

    # --- Розмір вхідного зображення ---
    imgsz=(img_properties['height'], img_properties['width']),  # (640, 640)

    # --- Параметри навчання ---
    epochs=CFG.EPOCHS,           # Кількість епох (70 для повного тренування)
    batch=CFG.BATCH_SIZE,        # Розмір батчу (зменшити при нестачі GPU-пам'яті)
    optimizer=CFG.OPTIMIZER,     # 'auto': ultralytics підбирає автоматично
    lr0=CFG.LR,                  # Початковий learning rate: 0.001
    lrf=CFG.LR_FACTOR,           # Кінцевий lr = lr0 * lrf: 0.001 * 0.01 = 0.00001
    weight_decay=CFG.WEIGHT_DECAY,   # L2-регуляризація: 0.0005
    dropout=CFG.DROPOUT,             # Dropout: 0.0 (вимкнено)
    fraction=CFG.FRACTION,           # Частка датасету: 1.0 = всі дані
    patience=CFG.PATIENCE,           # Early stopping через 20 епох без покращення
    profile=CFG.PROFILE,             # Профілювання швидкості (False — вимкнено)
    label_smoothing=CFG.LABEL_SMOOTHING,  # Розм'якшення міток (0.0 — вимкнено)

    # --- Метадані ---
    name=f'{CFG.BASE_MODEL}_{CFG.EXP_NAME}',  # Назва директорії для збереження результатів
    seed=CFG.SEED,    # Seed для відтворюваності

    # --- Додаткові параметри ---
    val=True,         # Валідувати після кожної епохи
    amp=True,         # AMP (Automatic Mixed Precision): прискорює тренування на GPU
    exist_ok=True,    # Перезаписати директорію результатів, якщо вже існує
    resume=False,     # False = починати з нуля; True = продовжити з checkpoint
    device=[0, 1],       # [0] — перша GPU; [0, 1] — дві GPU; None — CPU
    verbose=False,    # False — мінімальний вивід у консоль
)

In [ ]:
import os
import shutil

run_dir = f"/kaggle/working/deep/runs/detect/{CFG.BASE_MODEL}_{CFG.EXP_NAME}/weights"

best_src = os.path.join(run_dir, "best.pt")
last_src = os.path.join(run_dir, "last.pt")

best_dst = "/kaggle/working/best.pt"
last_dst = "/kaggle/working/last.pt"

shutil.copy2(best_src, best_dst)
shutil.copy2(last_src, last_dst)

print("Saved:")
print(best_dst)
print(last_dst)

In [ ]:
# Після тренування — завантажуємо найкращі ваги
model = YOLO('/kaggle/working/best.pt')

# Експортуємо модель у формат ONNX
model.export(
    format='onnx',   # Цільовий формат: onnx, openvino, engine (TensorRT), tflite тощо
    imgsz=(img_properties['height'], img_properties['width']),  # Розмір входу
    half=False,      # False: FP32 (повна точність); True: FP16 (менший розмір, швидше)
    int8=False,      # False: не квантизувати до INT8
    simplify=False,  # False: не спрощувати граф ONNX (зберегти повну структуру)
    nms=False,       # False: NMS виконується поза ONNX (у постобробці)
)

In [ ]:
!ls -lh /kaggle/working/*.pt
!ls -lh /kaggle/working/*.onnx

In [26]:

!zip -j /kaggle/working/yolo_models.zip \
    /kaggle/working/best.pt \
    /kaggle/working/last.pt \
    /kaggle/working/best.onnx

  adding: best.pt (deflated 8%)
  adding: last.pt (deflated 8%)
  adding: best.onnx (deflated 18%)


In [27]:
!ls -lh /kaggle/working/yolo_models.zip

-rw-r--r-- 1 root root 387M Sep 14 10:58 /kaggle/working/yolo_models.zip


---
## КРОК 9. АНАЛІЗ РЕЗУЛЬТАТІВ ТРЕНУВАННЯ

> **Контекст:** модель натренована і збережена у `runs/detect/`. Тепер переглядаємо всі автоматично збережені метрики і графіки.

In [ ]:
# Збираємо шляхи до всіх зображень з результатами тренування.
# Фільтруємо файли з 'batch' — вони містять приклади детекцій, а не метрики.
results_paths = [
    i for i in
    glob.glob(f'{CFG.OUTPUT_DIR}runs/detect/{CFG.BASE_MODEL}_{CFG.EXP_NAME}/*.png') +
    glob.glob(f'{CFG.OUTPUT_DIR}runs/detect/{CFG.BASE_MODEL}_{CFG.EXP_NAME}/*.jpg')
    if 'batch' not in i  # Виключаємо зображення батчів
]

print('Знайдено файлів:', len(results_paths))
results_paths

In [ ]:
# Виводимо всі метрики по черзі у відсортованому порядку
for file in sorted(results_paths):
    print(file)  # Назва файлу як заголовок
    display_image(file, print_info=False, hide_axis=True)
    print('\n')

In [ ]:
# Запускаємо валідацію для отримання детальних метрик по класах
# model.val() повертає об'єкт з полями: box.p, box.r, box.map50, box.map50_95
# а також виводить таблицю: Class | Images | Instances | P | R | mAP50 | mAP50-95
results_val = model.val(
    data=os.path.join(CFG.OUTPUT_DIR, 'data.yaml'),  # Конфігурація датасету
    imgsz=(img_properties['height'], img_properties['width']),
    conf=0.457,   # Оптимальний поріг впевненості (з F1_curve: F1=0.87 at 0.457)
    device=None,  # None = CPU; [0] = GPU
    split='val',  # Оцінюємо на validation split
)

# Ключові метрики (averaged по всіх класах)
print(f'\nmAP@0.5:       {results_val.box.map50:.4f}')
print(f'mAP@0.50:0.95: {results_val.box.map:.4f}')
print(f'Precision:     {results_val.box.mp:.4f}')
print(f'Recall:        {results_val.box.mr:.4f}')

In [ ]:
# Виводимо приклад детекцій на валідаційному батчі
validation_results_paths = [
    i for i in
    glob.glob(f'{CFG.OUTPUT_DIR}runs/detect/{CFG.BASE_MODEL}_{CFG.EXP_NAME}/*.png') +
    glob.glob(f'{CFG.OUTPUT_DIR}runs/detect/{CFG.BASE_MODEL}_{CFG.EXP_NAME}/*.jpg')
    if 'val_batch' in i  # Відбираємо тільки val_batch файли
]

if len(validation_results_paths) >= 1:
    val_img_path = random.choice(validation_results_paths)  # Випадковий батч
    print(f'Відображаємо: {val_img_path}')
    display_image(val_img_path, print_info=False, hide_axis=True)
else:
    print('Файли val_batch не знайдено. Перевір CFG.OUTPUT_DIR і CFG.EXP_NAME.')